# Notebook 05 — 4-Config Experiments & Evaluation

| Config | Model | RAG |
|--------|-------|-----|
| A | Base Qwen2.5-3B-Instruct | No |
| B | Base Qwen2.5-3B-Instruct | Yes (top-3 parent chunks) |
| C | Fine-tuned (SFT LoRA) | No |
| D | Fine-tuned (SFT LoRA) | Yes (top-3 parent chunks) |

**Metrics:** BLEU (char-level), ROUGE-L, BERTScore-F1, Source Recall@5 (B/D)

> **Prerequisites:** Notebooks 01–04 complete. FAISS index built from `all_chunks.jsonl`.

## 0. Setup

In [1]:
from pathlib import Path
import os

if os.path.exists('/content/'):
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/NLP_Final/Source/')
else:
    BASE = Path('../')

print(f'Base: {BASE}')

Mounted at /content/drive
Base: /content/drive/MyDrive/NLP_Final/Source


In [2]:
%%capture
!pip install -U \
    transformers peft accelerate bitsandbytes torchao \
    sacrebleu rouge-score bert-score \
    faiss-cpu FlagEmbedding tqdm

## 1. Configuration & Paths

In [3]:
import os, json, torch, pickle
import numpy as np

MODEL_ID  = 'Qwen/Qwen2.5-3B-Instruct'
SFT_PATH  = str(BASE / 'models/sft_checkpoint')
INDEX_DIR = BASE / 'data/vector_store/faiss_index'
RESULTS   = BASE / 'results'
RESULTS.mkdir(exist_ok=True)

SYSTEM_PROMPT = (
    'Ban la tro ly AI cua Truong Dai hoc Ton Duc Thang (TDTU). '
    'Ban tra loi cac cau hoi ve quy che, quy dinh, chinh sach cua truong mot cach chinh xac va day du. '
    'Tra loi bang tieng Viet. Neu khong co du thong tin, hay noi ro dieu do.'
)

print(f'Model     : {MODEL_ID}')
print(f'SFT path  : {SFT_PATH}')
print(f'Index dir : {INDEX_DIR}')

Model     : Qwen/Qwen2.5-3B-Instruct
SFT path  : /content/drive/MyDrive/NLP_Final/Source/models/sft_checkpoint
Index dir : /content/drive/MyDrive/NLP_Final/Source/data/vector_store/faiss_index


## 2. Load Test Dataset

In [4]:
test_pairs = []
with open(BASE / 'data/qa_filtered/qa_test.jsonl', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            test_pairs.append(json.loads(line))

references = [p['answer']   for p in test_pairs]
questions  = [p['question'] for p in test_pairs]
print(f'Test pairs: {len(test_pairs)}')

Test pairs: 284


## 3. Load FAISS Retriever + Parent Map

**Parent-child strategy:**  
- Child chunks (small, ~256–800 tok) → FAISS embedding index  
- Parent chunks (raw `\n\n` paragraphs) → inject vào LLM prompt để có context đầy đủ

In [5]:
import faiss, numpy as np
from FlagEmbedding import BGEM3FlagModel

faiss_index = faiss.read_index(str(INDEX_DIR / 'index.faiss'))
with open(INDEX_DIR / 'index.pkl', 'rb') as f:
    index_chunks = pickle.load(f)

parent_map: dict[str, str] = {}
parents_path = BASE / 'data/chunks/parent_chunks.jsonl'
if parents_path.exists():
    with open(parents_path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                p = json.loads(line)
                parent_map[p['parent_chunk_id']] = p['text']

embedder = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)


def retrieve(query: str, k: int = 5) -> list[dict]:
    out   = embedder.encode([query], return_dense=True,
                            return_sparse=False, return_colbert_vecs=False)
    q_vec = np.array(out['dense_vecs'], dtype='float32')
    scores, indices = faiss_index.search(q_vec, k)
    return [
        {'chunk': index_chunks[i], 'score': float(scores[0][j])}
        for j, i in enumerate(indices[0])
        if i < len(index_chunks)
    ]


def get_parent_context(results: list[dict], top_n: int = 3) -> tuple[list[str], list[str], list[str]]:
    """
    Returns (ctx_texts, source_files, parent_chunk_ids) deduped by parent_chunk_id.
    """
    seen:    set[str]  = set()
    texts:   list[str] = []
    sources: list[str] = []
    pids:    list[str] = []
    for r in results:
        pid = r['chunk'].get('parent_chunk_id', '')
        if pid and pid in seen:
            continue
        if pid:
            seen.add(pid)
        ctx = parent_map.get(pid) or r['chunk']['text']
        texts.append(ctx)
        sources.append(r['chunk']['source_file'])
        pids.append(pid)
        if len(texts) >= top_n:
            break
    return texts, sources, pids


print(f'FAISS index : {faiss_index.ntotal} vectors  |  dim={faiss_index.d}')
print(f'Parent map  : {len(parent_map)} parents')
print(f'Embedder    : BAAI/bge-m3 (FP16)')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

FAISS index : 280 vectors  |  dim=1024
Parent map  : 232 parents
Embedder    : BAAI/bge-m3 (FP16)


In [6]:
print('Checking setup...')
if (BASE / 'models/sft_checkpoint/adapter_config.json').exists():
    print('SFT adapter : found')
else:
    print('SFT adapter : NOT found — run notebook 04 first')

Checking setup...
SFT adapter : found


## 4. Load Models (Colab GPU only)

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from dotenv import load_dotenv
import json as _json

load_dotenv(BASE / '.env')
HF_TOKEN = os.environ.get('HF_TOKEN')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
COMPUTE_DTYPE = (
    torch.bfloat16 if torch.cuda.get_device_properties(0).total_memory > 30e9
    else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

# Đọc base model từ adapter_config để đúng với model đã train
_adapter_cfg = _json.loads((BASE / 'models/sft_checkpoint/adapter_config.json').read_text())
_base_id     = _adapter_cfg.get('base_model_name_or_path', MODEL_ID)
print(f'Base model : {_base_id}')
print(f'HF_TOKEN   : {"set" if HF_TOKEN else "not set"}')

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(_base_id, trust_remote_code=True, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
if tokenizer.eos_token != '<|im_end|>':
    tokenizer.eos_token    = '<|im_end|>'
    tokenizer.eos_token_id = tokenizer.convert_tokens_to_ids('<|im_end|>')

print('Loading base model + SFT adapter...')
_base = AutoModelForCausalLM.from_pretrained(
    _base_id, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True, token=HF_TOKEN,
)
ft_model = PeftModel.from_pretrained(_base, SFT_PATH)
ft_model.eval()
ft_model.config.use_cache = True

def use_base():       ft_model.disable_adapter_layers(); torch.cuda.empty_cache()
def use_finetuned():  ft_model.enable_adapter_layers();  torch.cuda.empty_cache()

print('Models ready.')
print(f'VRAM: {torch.cuda.memory_allocated(0)/1e9:.2f} GB')

Base model : unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit
HF_TOKEN   : set
Loading tokenizer...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

Models ready.
VRAM: 2.48 GB


## 5. Inference Function

In [8]:
def build_prompt(question: str, context_texts: list[str] = None) -> str:
    if context_texts:
        ctx_str = '\n\n'.join(f'[{i+1}] {t}' for i, t in enumerate(context_texts))
        user_content = (
            f'Dua vao cac doan van ban sau tu quy che TDTU:\n\n'
            f'{ctx_str}\n\nCau hoi: {question}'
        )
    else:
        user_content = question
    return (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{user_content}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )

print('build_prompt() ready.')

build_prompt() ready.


## 6. Run All 4 Configurations

In [9]:
from tqdm import tqdm

CONFIGS = {
    'A': {'use_rag': False, 'use_adapter': False, 'label': 'Base LLM, no RAG'},
    'B': {'use_rag': True,  'use_adapter': False, 'label': 'Base LLM + RAG'},
    'C': {'use_rag': False, 'use_adapter': True,  'label': 'Fine-tuned, no RAG'},
    'D': {'use_rag': True,  'use_adapter': True,  'label': 'Fine-tuned + RAG'},
}

BATCH_SIZE = 32

all_predictions:  dict[str, list[str]]       = {}
all_parent_ids:   dict[str, list[list[str]]] = {}  # parent_chunk_ids retrieved

for cfg_name, cfg in CONFIGS.items():
    out_path = RESULTS / f'config_{cfg_name}_results.jsonl'

    if out_path.exists():
        preds, pid_per_q = [], []
        with open(out_path, encoding='utf-8') as f:
            for line in f:
                row = json.loads(line)
                preds.append(row['prediction'])
                pid_per_q.append(row.get('retrieved_parent_ids', []))
        all_predictions[cfg_name] = preds
        all_parent_ids[cfg_name]  = pid_per_q
        print(f'Config {cfg_name}: {len(preds)} cached '
              f'(has_parent_ids={any(p for p in pid_per_q)})')
        continue

    print(f'\n=== Config {cfg_name}: {cfg["label"]} ===')
    use_finetuned() if cfg['use_adapter'] else use_base()

    bs = BATCH_SIZE if not cfg['use_rag'] else max(4, BATCH_SIZE // 2)
    preds, pid_per_q = [], []

    for i in tqdm(range(0, len(test_pairs), bs), desc=f'Config {cfg_name} (bs={bs})'):
        batch = test_pairs[i:i + bs]
        batch_prompts, batch_pids = [], []

        for pair in batch:
            if cfg['use_rag']:
                results = retrieve(pair['question'], k=5)
                ctx_texts, _, ctx_pids = get_parent_context(results, top_n=3)
                batch_prompts.append(build_prompt(pair['question'], ctx_texts))
                batch_pids.append(ctx_pids)
            else:
                batch_prompts.append(build_prompt(pair['question']))
                batch_pids.append([])

        inputs = tokenizer(
            batch_prompts, return_tensors='pt',
            truncation=True, max_length=1024, padding=True,
        ).to(ft_model.device)

        with torch.no_grad():
            outputs = ft_model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.05,
            )

        for j, inp_ids in enumerate(inputs['input_ids']):
            gen = outputs[j][len(inp_ids):]
            preds.append(tokenizer.decode(gen, skip_special_tokens=True).strip())
            pid_per_q.append(batch_pids[j])

    all_predictions[cfg_name] = preds
    all_parent_ids[cfg_name]  = pid_per_q

    with open(out_path, 'w', encoding='utf-8') as f:
        for pair, pred, pids in zip(test_pairs, preds, pid_per_q):
            f.write(json.dumps({
                'id':                   pair.get('id'),
                'question':             pair['question'],
                'reference':            pair['answer'],
                'prediction':           pred,
                'retrieved_parent_ids': pids,
            }, ensure_ascii=False) + '\n')
    print(f'  Saved {len(preds)} -> {out_path}')

use_finetuned()
print('\nAll configs done.')

Config A: 284 cached (has_parent_ids=False)
Config B: 284 cached (has_parent_ids=True)
Config C: 284 cached (has_parent_ids=False)
Config D: 284 cached (has_parent_ids=True)

All configs done.


## 7. Compute Metrics

**Recall@5 — Parent-level:** hit khi `parent_chunk_id` của gold QA pair xuất hiện trong top-3 retrieved parent IDs.  
Chính xác hơn source-file level vì kiểm tra đúng đoạn văn được truy xuất.

In [10]:
from sacrebleu.metrics import BLEU
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

bleu_metric = BLEU(tokenize='char')
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)


def compute_bleu(preds, refs):
    return bleu_metric.corpus_score(preds, [refs]).score


def compute_rougeL(preds, refs):
    scores = [rouge.score(r, p)['rougeL'].fmeasure for p, r in zip(preds, refs)]
    return sum(scores) / len(scores)


def compute_bertscore(preds, refs):
    P, R, F1 = bert_score_fn(
        preds, refs,
        lang='vi', model_type='bert-base-multilingual-cased',
        batch_size=8, device=device, verbose=False,
    )
    return {'P': P.mean().item(), 'R': R.mean().item(), 'F1': F1.mean().item()}


def compute_recall_at_k(pairs, pid_per_q, k=5):
    """
    Parent-level Recall@k.
    Hit: parent_chunk_id của gold QA pair xuất hiện trong top-k retrieved parent IDs.
    232/232 parent_chunk_ids đã sync giữa qa_test và parent_chunks.jsonl.
    """
    valid = [
        (p, pids) for p, pids in zip(pairs, pid_per_q)
        if p.get('parent_chunk_id') and pids
    ]
    if not valid:
        return None
    hits = sum(1 for p, pids in valid if p['parent_chunk_id'] in pids[:k])
    return hits / len(valid)


print('Metric functions ready.')

Metric functions ready.


In [11]:
summary: dict = {}

for cfg_name, cfg in CONFIGS.items():
    print(f'Computing metrics for Config {cfg_name}...')
    preds   = all_predictions[cfg_name]
    pid_per = all_parent_ids[cfg_name]

    bleu    = compute_bleu(preds, references)
    rL      = compute_rougeL(preds, references)
    bscore  = compute_bertscore(preds, references)
    recall5 = compute_recall_at_k(test_pairs, pid_per) if cfg['use_rag'] else None

    summary[cfg_name] = {
        'label':        cfg['label'],
        'bleu':         round(bleu, 2),
        'rougeL':       round(rL * 100, 2),
        'bertscore_P':  round(bscore['P'] * 100, 2),
        'bertscore_R':  round(bscore['R'] * 100, 2),
        'bertscore_F1': round(bscore['F1'] * 100, 2),
        'recall_at_5':  round(recall5 * 100, 2) if recall5 is not None else 'N/A',
    }
    r5_str = f'{recall5*100:.1f}%' if recall5 is not None else 'N/A'
    print(f'  BLEU={bleu:.2f}  ROUGE-L={rL*100:.2f}  BERT-F1={bscore["F1"]*100:.2f}  '
          f'Recall@5(parent)={r5_str}')

print('\nAll metrics computed.')

Computing metrics for Config A...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  BLEU=26.18  ROUGE-L=37.38  BERT-F1=74.12  Recall@5(parent)=N/A
Computing metrics for Config D...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  BLEU=9.24  ROUGE-L=22.89  BERT-F1=62.76  Recall@5(parent)=64.4%

All metrics computed.


## 8. Results Table

In [12]:
w = 90
print('=' * w)
print(f'{"Config":<8} {"Description":<32} {"BLEU":>7} {"ROUGE-L":>8} {"BERT-F1":>8} {"Recall@5":>10}')
print('=' * w)
for cfg_name in ['A', 'B', 'C', 'D']:
    m = summary[cfg_name]
    print(f'{cfg_name:<8} {m["label"]:<32} {m["bleu"]:>7.2f} {m["rougeL"]:>8.2f} '
          f'{m["bertscore_F1"]:>8.2f} {str(m["recall_at_5"]):>10}')
print('=' * w)

Config   Description                         BLEU  ROUGE-L  BERT-F1   Recall@5
A        Base LLM, no RAG                   17.07    28.61    68.38        N/A
B        Base LLM + RAG                      8.15    21.01    61.38      64.44
C        Fine-tuned, no RAG                 26.18    37.38    74.12        N/A
D        Fine-tuned + RAG                    9.24    22.89    62.76      64.44


## 9. Save Evaluation Summary

In [13]:
SUMMARY_PATH = RESULTS / 'eval_summary.json'
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f'Saved -> {SUMMARY_PATH}')

Saved -> /content/drive/MyDrive/NLP_Final/Source/results/eval_summary.json


## 10. Human Evaluation Template

In [14]:
import csv, random

human_eval_samples = []
indices = random.sample(range(len(test_pairs)), min(100, len(test_pairs)))

for idx in indices:
    pair = test_pairs[idx]
    for cfg_name in ['A', 'B', 'C', 'D']:
        human_eval_samples.append({
            'question_id':        pair.get('id', f'q{idx}'),
            'question':           pair['question'],
            'config':             cfg_name,
            'prediction':         all_predictions[cfg_name][idx],
            'reference':          pair['answer'],
            'accuracy_1_5':       '',
            'completeness_1_5':   '',
            'fluency_1_5':        '',
            'conciseness_1_5':    '',
            'no_hallucination_1_5': '',
        })

HUMAN_EVAL_PATH = RESULTS / 'human_eval_template.csv'
with open(HUMAN_EVAL_PATH, 'w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=human_eval_samples[0].keys())
    writer.writeheader()
    writer.writerows(human_eval_samples)

print(f'Saved human eval template -> {HUMAN_EVAL_PATH}')
print(f'Total rows: {len(human_eval_samples)} (100 questions x 4 configs)')

Saved human eval template -> /content/drive/MyDrive/NLP_Final/Source/results/human_eval_template.csv
Total rows: 400 (100 questions x 4 configs)


## 11. Qualitative Sample Comparison

In [15]:
sample_indices = random.sample(range(len(test_pairs)), 3)

for idx in sample_indices:
    pair = test_pairs[idx]
    print('=' * 80)
    print(f'Q : {pair["question"]}')
    ref = pair['answer']
    print(f'Ref: {ref[:200]}...' if len(ref) > 200 else f'Ref: {ref}')
    print()
    for cfg_name in ['A', 'B', 'C', 'D']:
        pred = all_predictions[cfg_name][idx]
        print(f'[{cfg_name}] {pred[:200]}...' if len(pred) > 200 else f'[{cfg_name}] {pred}')
    print()

Q : Em nghe nói sinh viên là trung tâm, vậy cụ thể nghĩa là sao ạ?
Ref: Ừ đúng rồi! Quy chế nêu rõ sinh viên là trung tâm của các hoạt động giáo dục và đào tạo. Nhà trường sẽ đảm bảo điều kiện để các em thực hiện đầy đủ quyền và nhiệm vụ trong học tập, nghiên cứu và rèn l...

[A] Theo nguyên tắc "sinh viên là trung tâm" tại Trường ĐH Tôn Đức Thắng (TDTU), có nghĩa là nhà trường luôn đặt quyền lợi và nhu cầu của sinh viên lên hàng đầu trong mọi hoạt động giáo dục và đào tạo. Cụ...
[B] Theo quy chế của Trường Đại học Tôn Đức Thắng (TDTU), sinh viên được coi là trung tâm trong quá trình hoạt động của trường. Nghĩa là, sinh viên không chỉ là đối tượng thụ động mà còn là người chủ động...
[C] Chào em! Sinh viên là trung tâm trong mọi hoạt động của trường đó. Nghĩa là nhà trường luôn quan tâm và chăm sóc đến nhu cầu, nguyện vọng của sinh viên, đồng thời cũng tôn trọng ý kiến của sinh viên đ...
[D] Chào em! Sinh viên là trung tâm trong công tác sinh viên, nghĩa là họ là đối tượng chính được c

---
**Next steps:**
- Fill `human_eval_template.csv` with scores
- Run `06_rlhf_optional.ipynb` for RLHF extension